# ReFuelEU optimisation — results

The paper's figures, drawn from what `01_optimisation_runs.ipynb` leaves in
`results/` — the JSON outputs, and for section 3 the optimisation histories. No model
runs here, so this notebook is seconds, not hours.

Sections 1 and 6 need only the `main` case; 2 and 3 need `main`'s ten budgets; 4 needs
all four biomass cases; 5 needs `main` and `pess`. Each section says what is missing rather
than failing, so it is usable while the sweeps are still running.

**One conversion changed.** Legacy MFSPs were €/L and the figures divided them by 35 to
reach €/MJ. On `main` they are stored in €/MJ already, so that division is gone — see
`00_migration_validation.ipynb` §1 on why 35.3, not 35 or 35.2, is the number the paper
actually used.

## 0. Setup

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from scipy.interpolate import griddata

import optimisation_runs as R

warnings.filterwarnings("ignore")

BUDGETS = [2.0, 2.2, 2.4, 2.6, 2.8, 3.0, 3.2, 3.4, 3.6, 3.8]
YEARS = np.array(R.YEARS)

# Paper colours.
GREEN, BLUE, RED, GREY = "#7e9b59", "#092054", "#cb3629", "lightgrey"


def available(case, tag):
    """Path of a saved run, or None if that run has not been done yet."""
    stem = f"{tag}_{case}" if tag in ("fossil", "refueleu") else f"opt_{case}_{tag}"
    path = R.RESULTS_DIR / f"{stem}.json"
    return path if path.exists() else None


def collect(case, tags):
    """Load every run of `case` that exists, keyed by tag."""
    out = {}
    for tag in tags:
        path = available(case, tag)
        if path:
            out[tag] = R.load(path)
    missing = [t for t in tags if t not in out]
    if missing:
        print(f"{case}: not on disk yet -> {', '.join(missing)}")
    return out


ALL_TAGS = ["mincarb"] + [R.budget_tag(b) for b in BUDGETS]

## 1. One optimised run against the fossil BAU

The headline numbers of §4.1, then Figure 6: the mandate the optimiser chose, and how
close each pathway sits to the two limits that bound it.

In [ ]:
RUN = "2_8"  # the case discussed in the paper's text

opt, _ = R.load(R.RESULTS_DIR / f"opt_main_{RUN}.json")
bau, _ = R.load(R.RESULTS_DIR / "fossil_main.json")

pd.DataFrame(
    [
        [
            "Airline cost per RPK",
            opt["total_cost_per_rpk"].loc[2050] / bau["total_cost_per_rpk"].loc[2050],
        ],
        ["Airfare per RPK", opt["airfare_per_rpk"].loc[2050] / bau["airfare_per_rpk"].loc[2050]],
        ["Traffic vs no elasticity", opt["rpk"].loc[2050] / opt["rpk_no_elasticity"].loc[2050]],
        ["Traffic vs BAU", opt["rpk"].loc[2050] / bau["rpk"].loc[2050]],
        [
            "Annual CO2 vs BAU",
            opt["co2_emissions_including_energy"].loc[2050]
            / bau["co2_emissions_including_energy"].loc[2050],
        ],
        [
            "Cumulative CO2 vs BAU",
            opt["cumulative_co2_emissions"].loc[2050] / bau["cumulative_co2_emissions"].loc[2050],
        ],
        [
            "RPK CAGR 2025-2050 (%)",
            ((opt["rpk"].loc[2050] / opt["rpk"].loc[2025]) ** (1 / 25) - 1) * 100,
        ],
        [
            "RPK CAGR 2025-2050, BAU (%)",
            ((bau["rpk"].loc[2050] / bau["rpk"].loc[2025]) ** (1 / 25) - 1) * 100,
        ],
    ],
    columns=["2050 indicator", "value"],
).round(4).to_string(index=False)

In [ ]:
# Figure 6 -- mandate, and each pathway against its ramp-up and resource envelopes.
fig, (ax_mandate, ax_bio, ax_ele) = plt.subplots(1, 3, figsize=(15, 5))

# --- left: the chosen mandate, against the regulation as written and as legislated ---
ax_mandate.plot(YEARS, opt["generic_biofuel_share_dropin_fuel"], color=GREEN, lw=2)
ax_mandate.plot(YEARS, opt["generic_electrofuel_share_dropin_fuel"], color=BLUE, lw=2)
ax_mandate.plot(YEARS, opt["fossil_kerosene_share_dropin_fuel"], color=RED, lw=2)

refueleu_years = [2020, 2025] + R.OPTIM_YEARS
refueleu_bio = [0, 2] + R.REFUELEU_MANDATE["biofuel"]
refueleu_ele = [0, 0] + R.REFUELEU_MANDATE["electrofuel"]
ax_mandate.plot(refueleu_years, refueleu_bio, "--", color=GREEN, lw=1.5)
ax_mandate.plot(refueleu_years, refueleu_ele, "--", color=BLUE, lw=1.5)
ax_mandate.plot(
    refueleu_years,
    [100 - b - e for b, e in zip(refueleu_bio, refueleu_ele)],
    "--",
    color=RED,
    lw=1.5,
)

# ReFuelEU as legislated: flat steps between reference years, not a linear ramp.
# Derived from the legislated totals (6 % SAF of which 1.2 % synthetic in 2030-34, and
# so on) rather than the hand-entered series the published figure carried.
step_bio = pd.Series(0.0, index=R.YEARS)
step_ele = pd.Series(0.0, index=R.YEARS)
for start, end, bio, ele in [
    (2025, 2030, 2, 0),
    (2030, 2035, 6, 1.2),
    (2035, 2040, 20, 5),
    (2040, 2045, 34, 10),
    (2045, 2050, 42, 15),
    (2050, 2051, 70, 35),
]:
    step_bio.loc[start : end - 1] = bio - ele
    step_ele.loc[start : end - 1] = ele
ax_mandate.plot(YEARS, step_bio, ":", color=GREEN, lw=1.5)
ax_mandate.plot(YEARS, step_ele, ":", color=BLUE, lw=1.5)
ax_mandate.plot(YEARS, 100 - step_bio - step_ele, ":", color=RED, lw=1.5)

ax_mandate.legend(
    handles=[
        plt.Line2D([], [], color=GREEN, lw=1.5, label="Biofuel"),
        plt.Line2D([], [], color=BLUE, lw=1.5, label="Electrofuel"),
        plt.Line2D([], [], color=RED, lw=1.5, label="Fossil"),
        plt.Line2D([], [], color="k", lw=1.5, ls="-", label="Optimisation"),
        plt.Line2D([], [], color="k", lw=1.5, ls="--", label="ReFuelEU (linear)"),
        plt.Line2D([], [], color="k", lw=1.5, ls=":", label="ReFuelEU (step)"),
    ],
    fontsize=10,
    loc="center left",
)
ax_mandate.set_ylabel("Drop-in fuel shares (%)")
ax_mandate.set_xlim(2021, 2050)
ax_mandate.set_ylim(0, 105)
ax_mandate.grid(alpha=0.3)
ax_mandate.set_title("Blending mandate")


def envelopes(consumption, rate=0.2, volume=0.2 * R.EU_ASK_SHARE):
    """The two ramp-up limits of Eq. 12, as the sawtooth the paper plots.

    Each reference year gets two points at the same abscissa: the cap implied by the
    previous year, then what the scenario actually consumed. The feasible set is the
    *larger* of the two caps, hence the max when shading.
    """
    years, caps_rate, caps_volume = (
        [2025],
        [consumption.loc[2025] / 1e12],
        [consumption.loc[2025] / 1e12],
    )
    for year in R.OPTIM_YEARS:
        previous = consumption.loc[year - 5]
        years += [year, year]
        caps_rate += [previous * (1 + rate) ** 5 / 1e12, consumption.loc[year] / 1e12]
        caps_volume += [(previous + volume * 5 * 1e12) / 1e12, consumption.loc[year] / 1e12]
    return years, caps_rate, caps_volume


for ax, pathway, origin, colour, title in [
    (ax_bio, "generic_biofuel", "biomass", GREEN, "Biofuel"),
    (ax_ele, "generic_electrofuel", "electricity", BLUE, "Electrofuel"),
]:
    consumption = opt[f"{pathway}_energy_consumption"]
    # Resource envelope in fuel terms: what the aviation allocation can actually make.
    resource_cap = (
        opt[f"{origin}_availability_aviation_allocated"] / R.RESOURCE_PER_FUEL[origin] / 1e12
    )

    years, caps_rate, caps_volume = envelopes(consumption)
    caps = np.maximum(caps_rate, caps_volume)

    ax.plot(YEARS, consumption / 1e12, color=colour, lw=3, label="Actual consumption")
    ax.plot(years, caps_rate, ":", color="#CCCCCC", lw=3, label="Ramp-up (rate)")
    ax.plot(years, caps_volume, "--", color="#CCCCCC", lw=2, label="Ramp-up (volume)")
    ax.plot(YEARS, resource_cap, "-", color="#CCCCCC", lw=2, label="Resource constraint")

    # Infeasible above either limit; feasible below both.
    ax.fill_between(YEARS, resource_cap, 3, facecolor=RED, alpha=0.1, lw=0)
    ax.fill_between(years, caps, 3, facecolor=RED, alpha=0.1, lw=0)
    ax.fill_between(
        years,
        0,
        np.minimum(caps, np.interp(years, YEARS, resource_cap)),
        facecolor=GREEN,
        alpha=0.1,
        lw=0,
    )

    ax.set_ylabel("Consumption [EJ]")
    ax.set_xlim(2025, 2050)
    ax.set_ylim(-0.05, 1.5)
    ax.grid()
    ax.legend(loc="upper left", fontsize=10)
    ax.set_title(title)

fig.tight_layout()
fig.savefig("ressource_constraints.pdf")

## 2. Carbon-budget sensitivity — the reference case

Figures 7 and 8: what the optimiser does with the mandate as the budget tightens, and
what that costs in traffic and airfare. The ReFuelEU point is drawn in red for scale.

In [ ]:
runs = collect("main", ALL_TAGS)
refueleu = R.load(R.RESULTS_DIR / "refueleu_main.json") if available("main", "refueleu") else None

ordered = [t for t in ALL_TAGS if t in runs]
labels = {"mincarb": "Min $CO_2$", **{R.budget_tag(b): f"{b}" for b in BUDGETS}}
cmap = plt.get_cmap("Blues")
colours = dict(zip(ordered, cmap(np.linspace(0.3, 1.0, len(ordered)))))

fig, axs = plt.subplots(2, 2, figsize=(12, 8), sharex="col")

panels = [
    (
        axs[0, 0],
        "generic_biofuel_share_dropin_fuel",
        "Biofuel Share (%)",
        "Biofuel Share",
        (-1, 59),
    ),
    (
        axs[1, 0],
        "generic_electrofuel_share_dropin_fuel",
        "Electrofuel Share (%)",
        "Electrofuel Share",
        (-1, 59),
    ),
    (axs[0, 1], "rpk", "RPK", "Traffic", None),
    (axs[1, 1], "airfare_per_rpk", "Airfare per RPK (€)", "Airfare", (0.06, 0.15)),
]

for ax, key, ylabel, title, ylim in panels:
    for tag in ordered:
        ax.plot(YEARS, runs[tag][0][key], color=colours[tag], lw=1.8)
    if refueleu is not None:
        ax.plot(YEARS, refueleu[0][key], color=RED, ls="--", lw=2, label="ReFuelEU")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xlim(2019)
    if ylim:
        ax.set_ylim(ylim)
    ax.grid(True)

# Two extra references: traffic without the price feedback, and the 2019 airfare that
# anchors both the elasticity and the inverse supply function.
axs[0, 1].plot(
    YEARS, runs[ordered[-1]][0]["rpk_no_elasticity"], color="black", ls="--", label="No elasticity"
)
axs[0, 1].set_ylim(0)
axs[1, 1].axhline(0.09236379319842411, color="black", ls="--", label="2019 airfare")

axs[0, 0].legend(loc="upper left", frameon=True)
axs[1, 0].legend(loc="upper left", frameon=True)
axs[0, 1].legend(loc="lower right", frameon=True)
axs[1, 1].legend(loc="upper left", frameon=True)
axs[1, 0].set_xlabel("Year")
axs[1, 1].set_xlabel("Year")

fig.subplots_adjust(right=0.82)
scalar = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=0, vmax=len(ordered) - 1))
cbar = fig.colorbar(scalar, cax=fig.add_axes([0.85, 0.15, 0.02, 0.7]))
cbar.set_label("Share of world carbon budget (%)")
cbar.set_ticks(range(len(ordered)))
cbar.set_ticklabels([labels[t] for t in ordered])

plt.tight_layout(rect=[0, 0, 0.82, 1])
plt.savefig("overall.pdf", bbox_inches="tight")

## 3. Which constraints bind, and when

No new model run: the constraint values at each optimum are already in the HDF that
`01_optimisation_runs.ipynb` saves beside every result. Five constraints, each enforced
at the five ReFuelEU reference years, plus the scalar carbon budget — 26 numbers per
run, negative for slack, zero for active, positive for violated.

The first figure is the raw value, the second reduces it to the state of each
constraint. The ramp-up constraints are normalised by their volume cap, so a pathway
using none of its allowance sits at exactly −1; the axis is clipped there.

In [ ]:
ROWS = [R.budget_tag(b) for b in sorted(BUDGETS, reverse=True)] + ["mincarb"]

constraints = {}
for tag in ROWS:
    hdf = R.RESULTS_DIR / f"opt_main_{tag}.hdf"
    read = R.read_constraints(hdf) if hdf.exists() else None
    if read is not None:
        constraints[tag] = read
missing = [tag for tag in ROWS if tag not in constraints]
if missing:
    print("no history on disk yet ->", ", ".join(missing))

shown = [tag for tag in ROWS if tag in constraints]

# A pathway held near zero has consumed none of its allowed ramp, which puts its
# constraint at exactly -1: the floor is an artefact of the normalisation, not a run
# that is unusually comfortable.
FLOOR = -1.05

cmap = plt.get_cmap("Blues")
budget_tags = [tag for tag in shown if tag != "mincarb"]
colours = dict(zip(budget_tags, cmap(np.linspace(0.35, 1.0, len(budget_tags)))))
colours["mincarb"] = GREEN

fig, axes = plt.subplots(1, len(R.CONSTRAINT_LABELS), figsize=(17, 3.8), sharey=True)
for ax, (name, pretty) in zip(axes, R.CONSTRAINT_LABELS.items()):
    for tag in shown:
        frame, _, feasible = constraints[tag]
        if name in frame:
            # A budget with no feasible solution is drawn dashed: the point plotted is
            # the optimiser's last iterate, not an optimum.
            ax.plot(
                frame.index,
                np.clip(frame[name], FLOOR + 0.02, None),
                "o-" if feasible else "o--",
                ms=3,
                lw=2.0 if tag == "mincarb" else 1.5,
                color=colours[tag],
                label=labels.get(tag, tag) + ("" if feasible else " (infeasible)"),
            )
    ax.axhline(0, color=RED, lw=1.2)
    ax.axhspan(FLOOR, 0, color="#f4f4f4", zorder=0)
    ax.set_title(pretty.replace("\n", " "), fontsize=10)
    ax.set_xticks(R.OPTIM_YEARS)
    ax.set_xticklabels([str(year)[2:] for year in R.OPTIM_YEARS])
    ax.set_xlabel("Enforcement year")
    ax.grid(alpha=0.3)

axes[0].set_ylabel("Constraint value (clipped at -1)")
axes[0].set_ylim(FLOOR, 0.25)
handles, names = axes[0].get_legend_handles_labels()
fig.legend(
    handles, names, title="Budget (Gt)", fontsize=8, loc="center left", bbox_to_anchor=(1.0, 0.5)
)
plt.tight_layout()
plt.savefig("constraints_by_budget.pdf", bbox_inches="tight")

In [ ]:
ACTIVE, APPROACHING = 1e-3, 0.05
STATES = [
    ("#f4f4f4", "Slack"),
    ("#bcd0e5", "Approaching (< 5 %)"),
    ("#092054", "Active"),
    ("#cb3629", "Violated"),
    ("#ffffff", "Not applicable"),
]

grid = np.full((len(shown), len(R.CONSTRAINT_LABELS) * len(R.OPTIM_YEARS) + 1), np.nan)
for row, tag in enumerate(shown):
    frame, carbon, _ = constraints[tag]
    for block, name in enumerate(R.CONSTRAINT_LABELS):
        if name in frame:
            grid[row, block * 5 : block * 5 + 5] = frame[name].values
    grid[row, -1] = carbon

state = np.full(grid.shape, 4.0)
state[grid < -APPROACHING] = 0
state[(grid >= -APPROACHING) & (grid < -ACTIVE)] = 1
state[np.abs(grid) <= ACTIVE] = 2
state[grid > ACTIVE] = 3
state[np.isnan(grid)] = 4

fig, ax = plt.subplots(figsize=(13, 4.2))
ax.imshow(
    state,
    aspect="auto",
    cmap=mcolors.ListedColormap([colour for colour, _ in STATES]),
    norm=mcolors.BoundaryNorm(np.arange(-0.5, len(STATES) + 0.5), len(STATES)),
)

ax.set_xticks(np.arange(grid.shape[1]))
ax.set_xticklabels([str(year)[2:] for year in R.OPTIM_YEARS] * len(R.CONSTRAINT_LABELS) + ["all"])
ax.set_yticks(np.arange(len(shown)))
ax.set_yticklabels([labels.get(tag, tag) for tag in shown])
ax.set_xlabel("Constraint enforcement year")
ax.set_ylabel("Carbon budget (Gt)")

# Cell borders, and a thicker rule between constraints.
ax.set_xticks(np.arange(grid.shape[1] + 1) - 0.5, minor=True)
ax.set_yticks(np.arange(len(shown) + 1) - 0.5, minor=True)
ax.grid(which="minor", color="white", lw=0.8)
ax.tick_params(which="minor", length=0)
for block in range(1, len(R.CONSTRAINT_LABELS) + 1):
    ax.axvline(block * 5 - 0.5, color="white", lw=3.5)
for block, pretty in enumerate(R.CONSTRAINT_LABELS.values()):
    ax.text(block * 5 + 2, -0.9, pretty, ha="center", va="bottom", fontsize=9)
ax.text(grid.shape[1] - 1, -0.9, "Carbon\nbudget", ha="center", va="bottom", fontsize=9)

# A run that ended infeasible has no optimum to read: mark the whole row.
for row, tag in enumerate(shown):
    if not constraints[tag][2]:
        ax.add_patch(
            plt.Rectangle(
                (-0.5, row - 0.5), grid.shape[1], 1, fill=False, hatch="///", edgecolor=RED, lw=0
            )
        )
        ax.get_yticklabels()[row].set_color(RED)

handles = [mpatches.Patch(facecolor=colour, edgecolor="0.7", label=name) for colour, name in STATES]
handles.append(
    mpatches.Patch(facecolor="white", edgecolor=RED, hatch="///", label="No feasible solution")
)
ax.legend(
    handles=handles,
    ncol=6,
    fontsize=8,
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.22),
)
plt.tight_layout()
plt.savefig("constraints_active_set.pdf")

The active set walks *backwards in time* as the budget tightens. At 3.8 Gt only the
biofuel ramp-up binds, and only in 2045; below 3.0 Gt the binding years are 2030–2035,
i.e. the scenario stops being limited by how much biomass exists and starts being
limited by how fast the plants can be built. At 2.0 Gt nothing satisfies the budget at
all — the row is hatched, and the violated constraint is the budget itself.

## 4. Biomass allocation — the trade-off surface

Figure 9. Every run of every biomass case is one point in (CO₂ consumed, biomass
allocated); the surface interpolated through them is the cost of the pair. The fossil
BAU appears at each allocation because it uses no biomass at all.

In [ ]:
BIOMASS_CASES = {"B5": 5.0, "B75": 7.5, "main": 9.9, "B15": 15.0}
CASE_COLOURS = {"main": "#cb3629", "B5": "#092054", "B15": "#efbd40", "B75": "#7e9b59"}

points = []
for case, biomass in BIOMASS_CASES.items():
    for tag, (vector, floats) in collect(case, ALL_TAGS).items():
        points.append(
            {
                "case": case,
                "biomass": biomass,
                "co2": floats["carbon_budget_consumed_share"] / R.EU_ASK_SHARE,
                "surplus": vector["cumulative_total_surplus_loss_discounted"].loc[2050] / 1e9,
            }
        )

# The BAU is biomass-free, so it is the same scenario at every allocation.
if available("main", "fossil"):
    bau_vector, bau_floats = R.load(R.RESULTS_DIR / "fossil_main.json")
    for biomass in BIOMASS_CASES.values():
        points.append(
            {
                "case": "fossil",
                "biomass": biomass,
                "co2": bau_floats["carbon_budget_consumed_share"] / R.EU_ASK_SHARE,
                "surplus": bau_vector["cumulative_total_surplus_loss_discounted"].loc[2050] / 1e9,
            }
        )

points = pd.DataFrame(points)
print(f"{len(points)} scenario points")
points.head()

In [ ]:
grid_x, grid_y = np.meshgrid(
    np.linspace(points["co2"].min(), points["co2"].max(), 200),
    np.linspace(points["biomass"].min(), points["biomass"].max(), 200),
)
grid_z = griddata(
    (points["co2"], points["biomass"]), points["surplus"], (grid_x, grid_y), method="cubic"
)

# Diverging scale centred on zero, so the sign of the surplus change is readable.
limit = 270
cmap = plt.get_cmap("RdBu_r")
norm = mcolors.Normalize(vmin=-limit, vmax=limit)

plt.figure(figsize=(10, 7))
mesh = plt.pcolormesh(grid_x, grid_y, grid_z, shading="gouraud", cmap=cmap, norm=norm)
contours = plt.contour(grid_x, grid_y, grid_z, levels=15, colors="black", linewidths=0.8)
for label in plt.clabel(contours, inline=True, fontsize=11, fmt="%.1f", colors="black"):
    label.set_path_effects([pe.withStroke(linewidth=3, foreground="white")])

plt.colorbar(mesh).set_label("Cumulative total surplus loss (Bn€, discounted)")
plt.scatter(
    points["co2"],
    points["biomass"],
    color=GREY,
    edgecolor="black",
    s=35,
    alpha=0.5,
    label="Scenario point",
    zorder=5,
)

if refueleu is not None:
    refueleu_surplus = refueleu[0]["cumulative_total_surplus_loss_discounted"].loc[2050] / 1e9
    refueleu_co2 = refueleu[1]["carbon_budget_consumed_share"] / R.EU_ASK_SHARE
    # The regulation implies its own biomass draw rather than choosing one; read it off
    # the run instead of placing the point by hand.
    refueleu_biomass = (
        refueleu[0]["generic_biomass_consumed_global_share"].loc[2050] * 100 / R.EU_ASK_SHARE / 100
    )
    plt.scatter(
        refueleu_co2,
        refueleu_biomass,
        s=100,
        color=cmap(norm(refueleu_surplus)),
        edgecolor="black",
        lw=1.2,
        zorder=10,
        label="ReFuelEU",
        path_effects=[pe.withStroke(linewidth=4, foreground="white")],
    )
    plt.text(
        refueleu_co2 + 0.012,
        refueleu_biomass + 0.12,
        f"ReFuelEU\n({refueleu_surplus:.1f} Bn€)",
        fontsize=11,
        color="purple",
        path_effects=[pe.withStroke(linewidth=3, foreground="white")],
        zorder=11,
    )

plt.xlabel("Carbon budget consumed share (%)")
plt.ylabel("Biomass allocated to aviation (%)")
plt.grid(True, ls="--")
plt.legend()
plt.tight_layout()
plt.savefig("2d_shares.pdf")

## 5. Pessimistic technology roadmap

Figure 10 and §4.3. Same problem with the drop-in efficiency gain cut from 1.35 to
0.91 %/yr — the only difference between `pess` and `main`. The dashed line is the extra
surplus loss attributable to the missing efficiency at each budget, which is the number
the section is really about.

In [ ]:
curves = {}
for case in ["main", "pess"]:
    rows = [
        {
            "co2": floats["carbon_budget_consumed_share"] / R.EU_ASK_SHARE,
            "surplus": vector["cumulative_total_surplus_loss_discounted"].loc[2050] / 1e9,
            "tag": tag,
        }
        for tag, (vector, floats) in collect(case, ALL_TAGS).items()
    ]
    if available(case, "fossil"):
        vector, floats = R.load(R.RESULTS_DIR / f"fossil_{case}.json")
        rows.append(
            {
                "co2": floats["carbon_budget_consumed_share"] / R.EU_ASK_SHARE,
                "surplus": vector["cumulative_total_surplus_loss_discounted"].loc[2050] / 1e9,
                "tag": "fossil",
            }
        )
    curves[case] = pd.DataFrame(rows).sort_values("co2")

plt.figure(figsize=(10, 6))
for case, colour, label in [("main", RED, "Reference"), ("pess", BLUE, "Low efficiency")]:
    plt.plot(curves[case]["co2"], curves[case]["surplus"], "-o", color=colour, label=label)
    for _, row in curves[case].iterrows():
        if row["tag"] in ("mincarb", "fossil"):
            plt.annotate(
                {"mincarb": "Min $CO_2$", "fossil": "Fossil"}[row["tag"]],
                (row["co2"], row["surplus"]),
                textcoords="offset points",
                xytext=(5, 5),
                ha="left",
            )

# Difference at matched CO2, extrapolating flat outside the low-efficiency range.
common = curves["main"]["co2"].values
delta = (
    np.interp(common, curves["pess"]["co2"], curves["pess"]["surplus"])
    - curves["main"]["surplus"].values
)
plt.plot(common[2:], delta[2:], "--o", color="black", label="Relative difference")

plt.xlabel("Carbon budget consumed share (%)")
plt.ylabel("Total and relative surplus loss (Bn€, discounted)")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig("technology.pdf")

In [ ]:
# Where the difference lands: on airlines, on passengers' wallets, or on traffic.
ref, _ = R.load(R.RESULTS_DIR / "opt_main_2_8.json")
low, _ = R.load(R.RESULTS_DIR / "opt_pess_2_8.json")

AIRFARE_2019 = 0.09236379319842411


def surplus_loss(v):
    """Annual passenger surplus loss relative to the 2019 price, in Bn€.

    The stored `area_loss` is the integral under the demand curve; the two correction
    terms move it from the counterfactual traffic to the realised one.
    """
    return (
        v["area_loss"]
        - AIRFARE_2019 * (v["rpk_no_elasticity"] - v["rpk"])
        + v["rpk"] * (v["airfare_per_rpk"] - AIRFARE_2019)
    ) / 1e9


panels = [
    (
        "Airline revenue (Bn€)",
        lambda v: (v["airfare_per_rpk"] - v["total_cost_per_rpk"]) * v["rpk"] / 1e9,
    ),
    ("Passenger spending (Bn€)", lambda v: v["airfare_per_rpk"] * v["rpk"] / 1e9),
    ("Passenger surplus loss (Bn€)", surplus_loss),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (title, metric) in zip(axes, panels):
    reference, degraded = metric(ref), metric(low)
    ax.plot(YEARS, reference, color=BLUE, label="Reference roadmap")
    ax.plot(YEARS, degraded, color=RED, label="Low efficiency")
    ax.fill_between(YEARS, reference, degraded, alpha=0.1, color=RED, label="Difference")
    cumulative = np.nansum(reference - degraded)
    ax.set_title(f"{title}\ncumulative Δ = {cumulative:.1f} Bn€")
    ax.set_xlabel("Year")
    ax.set_xlim(2020, 2050)
    ax.grid(True)
    ax.legend(fontsize=9)

plt.tight_layout()

## 6. Electrofuel abatement cost against the discount rate

Figure 11 — the only figure that needs no optimisation at all, since it compares the
pathways' own costs and emission factors. The surface is the electrofuel cost of
abatement discounted back to 2020; the dot-dashed lines mark where biofuel entering
service in a given year becomes the cheaper option.

Both quantities come from the *pathway* outputs, so this is a property of
`energy_rte.yaml` rather than of any run — a useful check that the carriers were
migrated correctly.

In [ ]:
prices = R.load(R.RESULTS_DIR / "refueleu_main.json")[0]

# MFSP is EUR/MJ on main; emission factors gCO2/MJ -> tCO2/MJ.
delta_price = {
    "biofuel": (prices["generic_biofuel_mean_mfsp"] - prices["fossil_kerosene_mean_mfsp"]).loc[
        2020:2050
    ],
    "electrofuel": (
        prices["generic_electrofuel_mean_mfsp"] - prices["fossil_kerosene_mean_mfsp"]
    ).loc[2020:2050],
}
delta_emissions = {
    "biofuel": (
        (
            prices["fossil_kerosene_mean_co2_emission_factor"]
            - prices["generic_biofuel_mean_co2_emission_factor"]
        )
        / 1e6
    ).loc[2020:2050],
    "electrofuel": (
        (
            prices["fossil_kerosene_mean_co2_emission_factor"]
            - prices["generic_electrofuel_mean_co2_emission_factor"]
        )
        / 1e6
    ).loc[2020:2050],
}

years_fine = np.arange(2020, 2050.05, 0.01)
rates = np.linspace(0.0, 0.2, 4000)
years_int = np.arange(2020, 2051)


def interpolate(series):
    return np.interp(years_fine, years_int, series.values)


# Electrofuel CAC, discounted from its entry-into-service year back to 2020.
cac = interpolate(delta_price["electrofuel"])[:, None] / (
    interpolate(delta_emissions["electrofuel"])[:, None]
    * (1 + rates[None, :]) ** (years_fine - 2020)[:, None]
)

plt.figure(figsize=(10, 5))
image = plt.imshow(
    np.clip(cac, 0, None),
    aspect="auto",
    origin="lower",
    extent=[0, 20, years_fine[0], years_fine[-1]],
    cmap="RdBu_r",
    interpolation="nearest",
    vmin=0,
    vmax=1500,
)

# Where electrofuel is *more* emissive than fossil the ratio is meaningless.
plt.contourf(
    rates * 100, years_fine, cac, levels=[cac.min(), 0], colors=RED, alpha=0.5, hatches=["///"]
)
plt.text(
    7,
    2023,
    "E-fuel more emissive than fossil",
    color=RED,
    fontsize=13,
    bbox=dict(facecolor="white", edgecolor=RED, boxstyle="round,pad=0.3"),
)

for entry_year in [2025, 2030, 2035, 2040]:
    threshold = np.interp(entry_year, years_int, delta_price["biofuel"]) / (
        np.interp(entry_year, years_int, delta_emissions["biofuel"])
        * (1 + rates) ** (entry_year - 2020)
    )
    crossing = np.full_like(rates, np.nan, dtype=float)
    for j in range(len(rates)):
        below = np.where((cac[:, j] <= threshold[j]) & (cac[:, j] >= 0))[0]
        if below.size:
            crossing[j] = years_fine[below[0]]
    plt.plot(rates * 100, crossing, color="white", lw=0.8, ls="-.")
    valid = np.where(~np.isnan(crossing))[0]
    if valid.size:
        plt.text(
            rates[valid[-1]] * 100 - 1.8,
            crossing[valid[-1]] + 0.9,
            f"EIS: {entry_year}",
            color="white",
            fontsize=9,
            va="center",
            rotation=-5,
        )

masked = np.ma.masked_where((cac <= 0) | (cac >= 1600), cac)
# Only the widely-spaced levels carry labels: the CAC blows up as the emission benefit
# goes to zero, so every contour above ~600 is pinched into the same corner and their
# labels land on top of each other.
labelled = [100, 200, 300, 400, 600]
iso = plt.contour(
    rates * 100,
    years_fine,
    masked,
    levels=labelled,
    colors="dimgrey",
    linewidths=0.8,
    linestyles="--",
)
for label in plt.clabel(iso, inline=True, fontsize=8, fmt=lambda x: f"{int(x)}"):
    label.set_color("white")
    label.set_path_effects([pe.withStroke(linewidth=1, foreground="dimgrey")])
plt.contour(
    rates * 100,
    years_fine,
    masked,
    levels=[lvl for lvl in np.arange(100, 1600, 100) if lvl not in labelled],
    colors="lightgrey",
    linewidths=0.5,
    linestyles=":",
)

plt.plot([], [], color="dimgrey", lw=0.8, ls="--", label="Electrofuel iso-CAC")
plt.plot([], [], color="white", lw=0.8, ls="-.", label="Below: biofuel is cheaper")
plt.legend(loc="upper right", fontsize=10, facecolor="white", edgecolor="white")

plt.colorbar(image).set_label(r"Electrofuel CAC (€/tCO$_2$), discounted to 2020")
plt.xlabel("Discount rate (%)")
plt.ylabel("Entry into service")
plt.tight_layout()
plt.savefig("sensitivity.pdf")

## Not migrated

`supply.pdf` comes from `equilibriums.ipynb`, not from `main.ipynb`. It is the
demand/supply calibration that produced `initial_airfare_per_rpk = 0.09236379319842411`
and depends on `IATA_cost_data.xlsx` rather than on any AeroMAPS model, so the migration
does not touch it and the legacy notebook still runs as written.

`discount_effect.pdf`, `biomass_sensitivity.pdf` and the `overall_colorbar*.pdf` variants
are stale artefacts in the legacy folder: no cell of `main.ipynb` writes them any more.